# **YouTube RAG chatbot**

In [88]:
!pip install -q \
langchain \
langchain-community \
langchain-groq \
youtube-transcript-api \
faiss-cpu \
sentence-transformers \
langchain-text-splitters

In [89]:
import os
from google.colab import userdata # Import userdata to access secrets

from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

# Retrieve the API key from Colab secrets and set it as an environment variable
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("GROQ_API_KEY successfully loaded from Colab secrets.")
except userdata.exceptions.KeyNotFoundError:
    print("GROQ_API_KEY not found in Colab secrets. Please add it to your secrets manager.")
except Exception as e:
    print(f"An error occurred while loading GROQ_API_KEY: {e}")

GROQ_API_KEY successfully loaded from Colab secrets.


In [90]:
video_id = "FwOTs4UxQS4"

In [91]:
api = YouTubeTranscriptApi()

transcript_data = api.fetch(video_id, languages=["en"])

transcript = " ".join(
    [chunk.text for chunk in transcript_data]
)

print(transcript[:1000])

AI. AI. AI. AI. AI. AI. You know, more agentic. Agentic capabilities. An AI agent. Agents. Agentic workflows. Agents. Agents. Agent. Agent. Agent. Agent. Agentic. All right. Most explanations of AI agents is either too technical or too basic. This video is meant for people like myself. You have zero technical background, but you use AI tools regularly and you want to learn just enough about AI agents to see how it affects you. In this video, we'll follow a simple one, two, three learning path by building on concepts you already understand like chatbt and then moving on to AI workflows and then finally AI agents. All the while using examples you will actually encounter in real life. And believe me when I tell you those intimidating terms you see everywhere like rag, rag, or react, they're a lot simpler than you think. Let's get started. Kicking things off at level one, large language models. Popular AI chatbots like CHBT, Google Gemini, and Claude are applications built on top of large 

In [92]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=150
)

chunks = splitter.create_documents([transcript])

In [93]:
len(chunks)

21

In [94]:
print(chunks[0].page_content)

AI. AI. AI. AI. AI. AI. You know, more agentic. Agentic capabilities. An AI agent. Agents. Agentic workflows. Agents. Agents. Agent. Agent. Agent. Agent. Agentic. All right. Most explanations of AI agents is either too technical or too basic. This video is meant for people like myself. You have zero technical background, but you use AI tools regularly and you want to learn just enough about AI agents to see how it affects you. In this video, we'll follow a simple one, two, three learning path by building on concepts you already understand like chatbt and then moving on to AI workflows and then


In [95]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [96]:
vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

In [97]:
vector_store.index_to_docstore_id

{0: '9267928c-d659-401f-9296-e93b01d1d7c8',
 1: 'c6250f28-581b-4c3e-9b28-66882ed9fbb9',
 2: '89f3b882-c0cb-47a2-9c05-8436b88d7886',
 3: 'e2607557-a1bf-4ed8-adfa-9aa45bddf5d4',
 4: 'fed26349-fc41-40b5-93f4-6f6d228c9f3c',
 5: '465513a3-2487-45b7-9fe9-6a35e548b337',
 6: '67c675ec-8635-4c45-96b1-daf22b55c26c',
 7: 'b2582f91-f7fe-42c5-a8b1-9412582be2e1',
 8: 'a6a42928-23b6-42ab-ae40-e817eb936f34',
 9: 'ad954af8-8e45-4a90-893b-bd9b544e94ab',
 10: 'e8687cf6-570a-4d86-a263-3ca8c5995953',
 11: '99132c89-8ccd-46f8-af62-513cddba9578',
 12: '0d75fb4b-f806-420a-9294-d0cbef39cf0b',
 13: '53f4a184-f788-44f3-9ad8-f9bda01a33a0',
 14: '3aef6abf-66fc-41a0-82ba-e224ea46cde7',
 15: '7f97795f-73cd-49ca-b507-df87d3a845b7',
 16: 'ba594219-871f-4320-8c01-73aba3038b1a',
 17: 'eee39bc9-0a4d-423a-ba99-cd3c69748aa9',
 18: 'a3745d5d-ada3-4768-8443-00666e5b6eb1',
 19: 'aac61c57-85fa-48dc-a235-9f8d55c15a39',
 20: 'e60fa1fa-3ea4-44a9-9372-da40cf792244'}

In [98]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [99]:
question = "What is this video about?"

In [100]:
retrieved_docs = retriever.invoke(question)

In [101]:
for doc in retrieved_docs:
    print(doc.page_content)
    print("="*80)

it's acting by looking at clips in video footage, trying to identify what it thinks a skier is, indexing that clip, and then returning that clip to us. Although this might not feel impressive, remember that an AI agent did all that instead of a human reviewing the footage beforehand, manually identifying the skier, and adding tags like skier, mountain, ski, snow. The programming is obviously a lot more technical and complicated than what we see in the front end, but that's the point of this demo, right? The average user like myself wants a simple app that just works without me having to
That was a hypothetical example. So let's move on to a real world AI agent example. Andrew is a preeeminent figure in AI and he created this demo website that illustrates how an AI agent works. I'll link the full video down below, but when I search for a keyword like skier, enter the AI vision agent in the background is first reasoning what a skier looks like. A person on skis going really fast in snow,

In [102]:
prompt = PromptTemplate(
    template="""
You are a helpful assistant.

Answer ONLY using the context below.

If the answer is not found in the context,
say "I don't know."

Context:
{context}

Question:
{question}
""",
    input_variables=["context","question"]
)

In [103]:
context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

In [104]:
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

In [105]:
final_prompt = prompt.invoke(
    {
        "context": context,
        "question": question
    }
)

In [106]:
response = llm.invoke(final_prompt)

In [107]:
print(response.content)

This video is about how AI agents work, with examples of a demo website that illustrates AI vision and the process of creating a basic AI agent.


## Interactive UI for YouTube Q&A

In [108]:
from IPython.display import display
import ipywidgets as widgets

# Create input widgets
video_id_input = widgets.Text(
    value='FwOTs4UxQS4', # Default video_id
    placeholder='Enter YouTube Video ID',
    description='Video ID:',
    disabled=False
)

question_input = widgets.Textarea(
    value='What is this video about?',
    placeholder='Enter your question here',
    description='Question:',
    disabled=False
)

run_button = widgets.Button(description="Get Answer")
output_widget = widgets.Output()

display(video_id_input, question_input, run_button, output_widget)

# Define the function to run when the button is clicked
def on_button_click(b):
    with output_widget:
        output_widget.clear_output()
        current_video_id = video_id_input.value
        current_question = question_input.value

        if not current_video_id:
            print("Please enter a YouTube Video ID.")
            return
        if not current_question:
            print("Please enter a question.")
            return

        try:
            # Fetch transcript
            api = YouTubeTranscriptApi()
            transcript_data = api.fetch(current_video_id, languages=["en"])
            transcript = " ".join([chunk.text for chunk in transcript_data])

            # Split into chunks
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=600,
                chunk_overlap=150
            )
            chunks = splitter.create_documents([transcript])

            # Create embeddings and vector store
            embeddings = HuggingFaceEmbeddings(
                model_name="sentence-transformers/all-MiniLM-L6-v2"
            )
            vector_store = FAISS.from_documents(
                chunks,
                embeddings
            )

            # Retrieve relevant documents
            retriever = vector_store.as_retriever(
                search_type="similarity",
                search_kwargs={"k":3}
            )
            retrieved_docs = retriever.invoke(current_question)

            # Prepare context for LLM
            context = "\n\n".join(
                doc.page_content
                for doc in retrieved_docs
            )

            # Invoke LLM
            llm = ChatGroq(
                model_name="llama-3.3-70b-versatile",
                temperature=0
            )
            prompt = PromptTemplate(
                template="""
You are a helpful assistant.

Answer ONLY using the context below.

If the answer is not found in the context,
say "I don't know."

Context:
{context}

Question:
{question}
""",
                input_variables=["context","question"]
            )
            final_prompt = prompt.invoke(
                {
                    "context": context,
                    "question": current_question
                }
            )
            response = llm.invoke(final_prompt)

            print("Answer:")
            print(response.content)

        except Exception as e:
            print(f"An error occurred: {e}")

# Attach the function to the button's on_click event
run_button.on_click(on_button_click)

Text(value='FwOTs4UxQS4', description='Video ID:', placeholder='Enter YouTube Video ID')

Textarea(value='What is this video about?', description='Question:', placeholder='Enter your question here')

Button(description='Get Answer', style=ButtonStyle())

Output()

In [ ]:
 git commit -m "Add: Colab YouTube-RAG-chatbot" git remote add origin https://github.com/rachit-barapatre/YouTube-RAG-chatbot.git git push -u origin main